In [ ]:
# 🚨🚨 EMERGENCY SETUP: FORCE OFFLINE MODE 🚨🚨
# Run this cell FIRST to prevent network timeouts

import os
os.environ['TIMM_CACHE_OFFLINE'] = '1'  # Force timm offline
os.environ['HF_HUB_OFFLINE'] = '1'      # Force HuggingFace offline
print("🚨 FORCED OFFLINE MODE: No network downloads allowed")
print("🚀 Ready for NUCLEAR speed training")

# Master Orchestration: Lazarus Training Mission
Immerse yourself in a guided control room that coordinates every model run, documents the artifacts, and keeps the laptop-friendly guardrails engaged.

## Mission Flow
1. Establish mission parameters and confirm the hardware envelope.
2. Load the global configuration and craft dataloaders sourced strictly from `./Data`.
3. Execute head-only warmups, optional fine-tunes, explainability exports, and experiment indexing.
4. Archive every artifact in the canonical structure for dashboard discovery.
5. Validate the pipeline with a fast smoke simulation before committing to longer voyages.

In [ ]:
# === SMART COMPROMISE: ACCURACY vs SPEED ===
# 🎯 BALANCED: ~70-75% accuracy in 1-2 hours

selected_models = ["efficientnet_b0"]  # Single model only
image_size_override = 160              # Balanced: 160px (vs 224)
batch_size_override = 24               # Balanced: Large but not huge  
num_epochs_override = 3                # SMART: 3 epochs (vs 1)
freeze_backbone_default = False        # SMART: Allow backbone fine-tuning
fast_test_mode = False                 # SMART: Use more data (20% vs 5%)

print("🎯 SMART COMPROMISE MODE: ACCURACY vs SPEED")
print("=" * 55)
print(f"   Model: {selected_models[0]}")
print(f"   Image size: {image_size_override} (balanced)")
print(f"   Batch size: {batch_size_override}")
print(f"   Epochs: {num_epochs_override} (compromise)")
print(f"   Backbone frozen: {freeze_backbone_default} (FINE-TUNE)")
print(f"   Fast test: {fast_test_mode} (MORE DATA)")
print("=" * 55)
print("⏱️  Expected time: 1-2 hours")
print("📈 Expected accuracy: ~70-75%")
print("💾 Models save to: ./models/")
print("📊 Dashboard ready for integration")

# ACCURACY BOOSTER: Use transfer learning if available
use_pretrained_fallback = True
print("🚀 Will attempt pretrained weights with offline fallback")

In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path
import textwrap
import torch
import psutil
from datetime import datetime, timezone
from typing import Any

# Fix import path for notebook execution
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    
# Add parent directory to path for src imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.master_trainer import MasterTrainer
from src.data_utils_torch import make_dataloaders

DATA_ROOT = PROJECT_ROOT / "Data"
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Data root: {DATA_ROOT}")
print(f"✓ Config path: {CONFIG_PATH}")

In [ ]:
def describe_hardware() -> dict[str, Any]:
    snapshot: dict[str, Any] = {"torch_version": torch.__version__}
    snapshot["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        snapshot["cuda_device"] = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        snapshot["cuda_total_memory_gb"] = round(props.total_memory / 1e9, 2)
        snapshot["cuda_free_memory_gb"] = round(torch.cuda.mem_get_info()[0] / 1e9, 2)
    virtual_mem = psutil.virtual_memory()
    snapshot["system_ram_gb"] = round(virtual_mem.total / 1e9, 2)
    snapshot["system_ram_available_gb"] = round(virtual_mem.available / 1e9, 2)
    snapshot["recommended_batch"] = 8 if virtual_mem.total >= 12 * 1024**3 else 4
    snapshot["recommended_image_size"] = 224
    snapshot["timestamp"] = datetime.now(timezone.utc).isoformat()
    return snapshot

hardware_report = describe_hardware()
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Data folder not found at '{DATA_ROOT}'. "
        "Please place dataset in root/Data with class subdirectories."
    )

# Count dataset classes
class_dirs = [d for d in DATA_ROOT.iterdir() if d.is_dir() and not d.name.startswith('.')]
total_images = sum(len(list(d.glob('*.jpg')) + list(d.glob('*.png'))) for d in class_dirs)

print("=" * 60)
print("🚀 LAZARUS MISSION CONTROL - HARDWARE PROFILE")
print("=" * 60)
print(f"  PyTorch version:     {hardware_report['torch_version']}")
print(f"  CUDA available:      {hardware_report['cuda_available']}")
if hardware_report['cuda_available']:
    print(f"  GPU device:          {hardware_report['cuda_device']}")
    print(f"  GPU memory (total):  {hardware_report['cuda_total_memory_gb']} GB")
    print(f"  GPU memory (free):   {hardware_report['cuda_free_memory_gb']} GB")
print(f"  System RAM (total):  {hardware_report['system_ram_gb']} GB")
print(f"  System RAM (free):   {hardware_report['system_ram_available_gb']} GB")
print(f"  Dataset classes:     {len(class_dirs)}")
print(f"  Total images:        {total_images:,}")
print()
print(f"  ⚙️  Recommended batch size:  {hardware_report['recommended_batch']}")
print(f"  ⚙️  Recommended image size:  {hardware_report['recommended_image_size']}")
print("=" * 60)

In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    global_config = json.loads(json.dumps(__import__('yaml').safe_load(handle)))

suite_config = global_config.get("training_suite", {})
requested_models = selected_models or [entry.get('name') for entry in global_config.get('models', [])]
requested_models = [name for name in requested_models if name]

print('Resolved model itinerary:', requested_models)
print('Fast test mode:', fast_test_mode)

In [ ]:
trainer = MasterTrainer(
    config_path=CONFIG_PATH,
    data_root=DATA_ROOT
)
training_kwargs = {}
if batch_size_override is not None:
    training_kwargs['batch_size_override'] = batch_size_override
if image_size_override is not None:
    training_kwargs['image_size_override'] = image_size_override
if num_epochs_override is not None:
    training_kwargs['num_epochs_override'] = num_epochs_override
if freeze_backbone_default is not None:
    training_kwargs['freeze_backbone_default'] = freeze_backbone_default

mission_results = trainer.run(model_names=requested_models, fast_test=fast_test_mode)
print(f'Completed {len(mission_results)} model runs.')

In [ ]:
from pprint import pprint
pprint(mission_results)

In [ ]:
print('Initiating fast smoke validation on a classifier-head only subset.')
smoke_outcome = trainer.run(model_names=requested_models[:1] if requested_models else None, fast_test=True)
print('Smoke validation complete:')
pprint(smoke_outcome)